In [1]:
import pandas as pd
import sqlite3
import os

In [6]:
txt_folder = r"../Data/raw/"
db_path = r"../Data/database/faers_2025.db"

os.makedirs(os.path.dirname(db_path), exist_ok=True)

In [7]:
# 2. Establish a connection to the SQLite database
conn = sqlite3.connect(db_path)

# Define the 7 core tables and the 4 quarters of the year
tables = ["DEMO", "DRUG", "REAC", "THER", "OUTC", "RPSR", "INDI"]
quarters = ["Q1", "Q2", "Q3", "Q4"]
year = "25"

print(" Starting Data Ingestion Pipeline...")

for table_name in tables:
    print(f"\n--- Processing Table: {table_name} ---")
    
    for q in quarters:
        file_name = f"{table_name}{year}{q}.txt"
        file_path = os.path.join(txt_folder, file_name)
        
        if os.path.exists(file_path):
            print(f"Loading {file_name}...")
            
            try:
                # Read the file safely, bypassing memory warnings and skipping corrupted lines
                df = pd.read_csv(file_path, sep="$", low_memory=False, on_bad_lines='skip')
                
                # --- Universal Cleaning ---
                # 1. Standardize column names to lowercase for SQL compatibility
                df.columns = df.columns.str.lower()
                
                # 2. Drop columns that are entirely empty to save storage
                df.dropna(axis=1, how='all', inplace=True)
                
                # 3. Inject the cleaned dataframe into the SQLite database
                df.to_sql(table_name.lower(), conn, if_exists='append', index=False)
                print(f"✅ Inserted {len(df)} rows into '{table_name.lower()}' table.")
                
            except Exception as e:
                print(f" Error processing {file_name}: {e}")
        else:
            print(f"Warning: {file_name} not found at {file_path}")

# Close the database connection to free up resources
conn.close()
print("\n Pipeline execution completed! Database 'faers_2025.db'")

 Starting Data Ingestion Pipeline...

--- Processing Table: DEMO ---
Loading DEMO25Q1.txt...
✅ Inserted 400514 rows into 'demo' table.
Loading DEMO25Q2.txt...
✅ Inserted 393130 rows into 'demo' table.
Loading DEMO25Q3.txt...
✅ Inserted 438512 rows into 'demo' table.
Loading DEMO25Q4.txt...
✅ Inserted 385288 rows into 'demo' table.

--- Processing Table: DRUG ---
Loading DRUG25Q1.txt...
✅ Inserted 2008162 rows into 'drug' table.
Loading DRUG25Q2.txt...
✅ Inserted 1829056 rows into 'drug' table.
Loading DRUG25Q3.txt...
✅ Inserted 2148451 rows into 'drug' table.
Loading DRUG25Q4.txt...
✅ Inserted 1815349 rows into 'drug' table.

--- Processing Table: REAC ---
Loading REAC25Q1.txt...
✅ Inserted 1432926 rows into 'reac' table.
Loading REAC25Q2.txt...
✅ Inserted 1340666 rows into 'reac' table.
Loading REAC25Q3.txt...
✅ Inserted 1535133 rows into 'reac' table.
Loading REAC25Q4.txt...
✅ Inserted 1349105 rows into 'reac' table.

--- Processing Table: THER ---
Loading THER25Q1.txt...
✅ Inserted 